# Session 3: when one shift is not enough
**A. Real-world problem.** Real rain does not move as one rigid sheet. One storm may move east
while another, 60 km away, moves north-west; cells grow and fade; and we are never sure of the
wind. Today you build a nowcast with local motion vectors, measure how its skill falls with
lead time, see where advection fails, and give a probability instead of a single answer.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
N = 100               # grid size (pixels); 1 pixel = 1 km
DT_MIN = 5.0          # minutes per frame
THR = 10.0            # rain-rate threshold (mm/h) for yes/no scores in this session

## Step 1: scenes with known truth
A scene is a list of cells (cx, cy, sigma, peak, u, v, g): start position (pixels), width (pixels),
centre rain rate (mm/h), velocity (pixels per frame) and growth rate g per frame. After t frames
a cell is at (cx + u t, cy + v t) and has peak * (1 + g)^t. Growth g = 0 means pure translation.

In [ ]:
def scene_field(scene, t=0.0, n=N):
    y, x = np.mgrid[0:n, 0:n]
    f = np.zeros((n, n))
    for cx, cy, s, p, u, v, g in scene:
        f += p * (1 + g) ** t * np.exp(-((x - cx - u * t) ** 2 + (y - cy - v * t) ** 2) / (2 * s * s))
    return f


# west cells move east, east cells move north-west: two regions, two motions
TWO_REGION = [(12, 25, 5.0, 30.0, 2.0, 0.5, 0.0), (18, 70, 6.0, 25.0, 2.0, 0.5, 0.0),
              (82, 30, 5.0, 35.0, -1.5, 1.5, 0.0), (78, 75, 6.0, 30.0, -1.5, 1.5, 0.0)]
# the same motions, but cells grow (+8 % per frame) or fade (-10 % per frame)
GROWTH = [(12, 25, 5.0, 22.0, 2.0, 0.5, 0.08), (18, 70, 6.0, 30.0, 2.0, 0.5, -0.10),
          (82, 30, 5.0, 22.0, -1.5, 1.5, 0.08), (78, 75, 6.0, 30.0, -1.5, 1.5, -0.10)]
ONE_REGION = [(12, 25, 5.0, 30.0, 2.0, 0.5, 0.0), (18, 70, 6.0, 25.0, 2.0, 0.5, 0.0)]


def without_growth(scene):
    """The same cells and motions with growth switched off: what pure advection can at best reproduce."""
    return [(cx, cy, s, p, u, v, 0.0) for cx, cy, s, p, u, v, g in scene]

## Step 2: sampling a field at any position
Everything below needs "the value of the field at a non-integer position". We use bilinear
interpolation (Session 2). Outside the picture the rain is zero.

In [ ]:
def sample(field, sx, sy):
    """Bilinear value of `field` at positions (sx, sy) given as arrays of pixels; zero outside."""
    ny, nx = field.shape
    x0, y0 = np.floor(sx).astype(int), np.floor(sy).astype(int)
    fx, fy = sx - x0, sy - y0
    pad = np.pad(field, 1)

    def at(yy, xx):
        ok = (yy >= -1) & (yy <= ny) & (xx >= -1) & (xx <= nx)
        return np.where(ok, pad[np.clip(yy + 1, 0, ny + 1), np.clip(xx + 1, 0, nx + 1)], 0.0)

    return ((1 - fx) * (1 - fy) * at(y0, x0) + fx * (1 - fy) * at(y0, x0 + 1)
            + (1 - fx) * fy * at(y0 + 1, x0) + fx * fy * at(y0 + 1, x0 + 1))


def shift_field(field, dx, dy):
    y, x = np.mgrid[0:field.shape[0], 0:field.shape[1]].astype(float)
    return sample(field, x - dx, y - dy)


def rmse(f, o):
    return float(np.sqrt(np.mean((f - o) ** 2)))


def csi(f, o, thr=THR):
    a, b = f >= thr, o >= thr
    d = int((a & b).sum() + (~a & b).sum() + (a & ~b).sum())
    return float("nan") if d == 0 else float((a & b).sum() / d)

## Step 3: local motion by block matching
Cut the picture into overlapping blocks. For each block that contains rain, find the shift of
the old picture that best matches the new one *inside that block only*. Then spread the block
vectors over every pixel (empty blocks borrow the nearest block that has a vector).

In [ ]:
def match_block(f0, f1, y, x, size, search=6):
    """Best shift (u, v) in pixels/frame for the block f1[y:y+size, x:x+size] (whole pixels, then a parabola fit)."""
    yy, xx = np.mgrid[y:y + size, x:x + size].astype(float)
    tgt = f1[y:y + size, x:x + size]

    def err(du, dv):
        return float(((sample(f0, xx - du, yy - dv) - tgt) ** 2).sum())

    best = min(((err(du, dv), du, dv) for dv in range(-search, search + 1) for du in range(-search, search + 1)))
    _, u, v = best
    u, v = float(u), float(v)
    h = 0.5
    for _ in range(3):
        for axis in (0, 1):
            d = np.array([h, 0.0]) if axis == 0 else np.array([0.0, h])
            em, e0, ep = err(*(np.array([u, v]) - d)), err(u, v), err(*(np.array([u, v]) + d))
            curv = em - 2 * e0 + ep
            if curv > 1e-12:
                step = float(np.clip(h * (em - ep) / (2 * curv), -h, h))
                u, v = (u + step, v) if axis == 0 else (u, v + step)
        h /= 2
    return u, v


def global_motion(f0, f1, search=8):
    """One vector for the whole picture."""
    return match_block(f0, f1, 0, 0, f0.shape[0], search)


def local_vectors(f0, f1, size=32, stride=16, min_rain=THR / 2):
    """Block-centre coordinates and vectors; NaN where a block has no rain."""
    ys = list(range(0, f1.shape[0] - size + 1, stride))
    xs = list(range(0, f1.shape[1] - size + 1, stride))
    U = np.full((len(ys), len(xs)), np.nan)
    V = U.copy()
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            if f1[y:y + size, x:x + size].max() >= min_rain:
                U[i, j], V[i, j] = match_block(f0, f1, y, x, size)
    return np.array(xs) + (size - 1) / 2.0, np.array(ys) + (size - 1) / 2.0, U, V


def vector_field(xc, yc, U, V, shape=(N, N)):
    """Fill empty blocks with the nearest vector, then interpolate to every pixel. Returns (Upix, Vpix)."""
    U, V = U.copy(), V.copy()
    ok = ~np.isnan(U)
    if not ok.any():                      # no rain anywhere: no motion information, assume still air
        U[:], V[:] = 0.0, 0.0
        ok = np.ones_like(ok)
    idx = np.argwhere(ok)
    for i in range(U.shape[0]):
        for j in range(U.shape[1]):
            if not ok[i, j]:
                k = idx[np.argmin(((idx - [i, j]) ** 2).sum(axis=1))]
                U[i, j], V[i, j] = U[k[0], k[1]], V[k[0], k[1]]

    def up(A):
        rows = np.array([np.interp(np.arange(shape[1]), xc, r) for r in A])
        return np.array([np.interp(np.arange(shape[0]), yc, rows[:, k]) for k in range(shape[1])]).T

    return up(U), up(V)

## Step 4: advect with a vector field (semi-Lagrangian)
For each target pixel, look *backwards* along the motion to where its rain came from, and copy
the value from there (interpolated). Because the vector should be the one at the *departure*
point, we repeat the look-back a few times. One look-back covers the whole lead time.

In [ ]:
def advect(field, U, V, frames, iters=3):
    """Single-step semi-Lagrangian advection over `frames` frames. U, V: arrays or scalars (pixels/frame)."""
    ny, nx = field.shape
    y, x = np.mgrid[0:ny, 0:nx].astype(float)
    U = np.broadcast_to(np.asarray(U, dtype=float), field.shape)
    V = np.broadcast_to(np.asarray(V, dtype=float), field.shape)
    dx, dy = frames * U, frames * V
    for _ in range(iters):
        sx, sy = np.clip(x - dx, 0, nx - 1), np.clip(y - dy, 0, ny - 1)
        dx, dy = frames * sample(np.array(U), sx, sy), frames * sample(np.array(V), sx, sy)
    return sample(field, x - dx, y - dy)


def advect_repeated(field, u, v, steps):
    """The same total motion done as `steps` separate one-frame shifts (each one interpolates and blurs)."""
    out = field
    for _ in range(steps):
        out = shift_field(out, u, v)
    return out


def second_moment_x(field):
    x = np.arange(field.shape[1])
    col = field.sum(axis=0)
    m = (col * x).sum() / col.sum()
    return float((col * (x - m) ** 2).sum() / col.sum())

## Step 5: skill against lead time
Observe two frames (t = -1 and 0), estimate motion, forecast 1..12 frames ahead, score against the truth.

In [ ]:
def nowcast_curves(scene, thr=THR, leads=range(1, 13)):
    f0, f1 = scene_field(scene, -1), scene_field(scene, 0)
    gu, gv = global_motion(f0, f1)
    U, V = vector_field(*local_vectors(f0, f1))
    out = {"persistence": [], "global": [], "local": []}
    for k in leads:
        ob = scene_field(scene, k)
        out["persistence"].append(csi(f1, ob, thr))
        out["global"].append(csi(advect(f1, gu, gv, k), ob, thr))
        out["local"].append(csi(advect(f1, U, V, k), ob, thr))
    return out


def first_below(curve, level=0.5, leads=range(1, 13)):
    """First lead time in minutes at which the curve is below `level`, or None if it never is."""
    for k, c in zip(leads, curve):
        if c < level:
            return int(k * DT_MIN)
    return None

## Step 6: uncertainty, as a probability
The wind estimate is uncertain. Make an ensemble: many copies of the wind, each with a small random
error, advect once per member, and count the fraction of members that put rain >= threshold on a pixel.

In [ ]:
def exceedance_probability(field, u, v, frames, sigma_v, members, rng, thr=THR):
    acc = np.zeros(field.shape)
    for _ in range(members):
        du, dv = rng.normal(0, sigma_v, 2)
        acc += shift_field(field, (u + du) * frames, (v + dv) * frames) >= thr
    return acc / members


def brier(p, o):
    return float(np.mean((p - o) ** 2))


def reliability(p, o, edges=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0001)):
    """Per probability bin: (mean forecast probability, observed frequency, count)."""
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi)
        rows.append((float(p[m].mean()) if m.any() else float("nan"), float(o[m].mean()) if m.any() else float("nan"), int(m.sum())))
    return rows

## Step 7: failure cases
Advection moves rain that exists. It cannot create a new storm, and a straight-line shift cannot
follow a rotating pattern.

In [ ]:
def rotating_scene_fields(frames, centre=(50.0, 50.0), radius=22.0, deg_per_frame=5.0):
    """Four cells on a ring turning counter-clockwise; returns the field after `frames` frames."""
    f = np.zeros((N, N))
    y, x = np.mgrid[0:N, 0:N]
    for k in range(4):
        a = np.radians(90 * k + deg_per_frame * frames)
        cx, cy = centre[0] + radius * np.cos(a), centre[1] + radius * np.sin(a)
        f += 30.0 * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * 5.0 ** 2))
    return f


NEW_CELL = (55.0, 10.0, 5.0, 35.0)       # a storm that does not exist yet at t = 0


def with_new_storm(field, start_frame, t):
    """Add the new storm, growing linearly from 0 at start_frame to its full peak 3 frames later."""
    amp = float(np.clip((t - start_frame) / 3.0, 0.0, 1.0))
    return field + scene_field([(NEW_CELL[0], NEW_CELL[1], NEW_CELL[2], NEW_CELL[3] * amp, 0.0, 0.0, 0.0)], 0)

In [ ]:
def disc_overlap_csi(r_fc, r_ob, d):
    """CSI of two discs with radii r_fc (forecast) and r_ob (truth) and centres d apart (km)."""
    a1, a2 = np.pi * r_fc ** 2, np.pi * r_ob ** 2
    if a1 == 0 or a2 == 0:
        return 0.0
    if d >= r_fc + r_ob:
        ov = 0.0
    elif d <= abs(r_fc - r_ob):
        ov = min(a1, a2)
    else:
        c1 = (d * d + r_fc ** 2 - r_ob ** 2) / (2 * d * r_fc)
        c2 = (d * d + r_ob ** 2 - r_fc ** 2) / (2 * d * r_ob)
        ov = (r_fc ** 2 * np.arccos(c1) + r_ob ** 2 * np.arccos(c2)
              - 0.5 * np.sqrt((-d + r_fc + r_ob) * (d + r_fc - r_ob) * (d - r_fc + r_ob) * (d + r_fc + r_ob)))
    return float(ov / (a1 + a2 - ov))


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    f0, f1 = scene_field(TWO_REGION, -1), scene_field(TWO_REGION, 0)
    gu, gv = global_motion(f0, f1)
    xc, yc, U, V = local_vectors(f0, f1)
    print(f"global vector ({gu:.2f}, {gv:.2f}) px/frame; local vectors from {int((~np.isnan(U)).sum())} blocks with rain")

    fig, ax = plt.subplots(figsize=(5.5, 5.2))
    ax.imshow(f1, origin="lower", cmap="Blues", vmin=0, vmax=40)
    ax.quiver(xc[None, :].repeat(len(yc), 0), yc[:, None].repeat(len(xc), 1), U, V, color="red", angles="xy", scale_units="xy", scale=0.5)
    ax.quiver([50], [50], [gu], [gv], color="black", angles="xy", scale_units="xy", scale=0.5)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title("Local vectors (red) and the one global vector (black)")
    fig.tight_layout()
    fig.savefig(OUT / "motion_vectors.png", dpi=110)

    leads = np.arange(1, 13) * DT_MIN
    cases = {"one region": ONE_REGION, "two regions": TWO_REGION, "two regions + growth/decay": GROWTH}
    fig2, axs = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
    for ax, (name, sc) in zip(axs, cases.items()):
        cv = nowcast_curves(sc)
        for key, st in (("persistence", "o-"), ("global", "s-"), ("local", "^-")):
            ax.plot(leads, cv[key], st, label=key)
            print(f"{name:28s} {key:11s} CSI at 30 min {cv[key][5]:.2f}, at 60 min {cv[key][11]:.2f}")
        ax.axhline(0.5, color="gray", ls=":")
        ax.set_title(name)
        ax.set_xlabel("lead time (min)")
        ax.grid(alpha=0.3)
    axs[0].set_ylabel("CSI at 10 mm/h")
    axs[0].legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "skill_curves.png", dpi=110)

    cell = scene_field([(20, 50, 2.5, 30.0, 0, 0, 0)])
    rep = advect_repeated(cell, 0.7, 0.0, 12)
    one = shift_field(cell, 8.4, 0.0)
    print(f"peak: start {cell.max():.2f}, 12 small steps {rep.max():.2f}, one 60-min shift {one.max():.2f}")

    rng = np.random.default_rng(1)
    fg = scene_field(ONE_REGION, 0)
    P, Pn, O = [], [], []
    for _ in range(40):
        du, dv = rng.normal(0, 0.35, 2)
        o = shift_field(fg, (2.0 + du) * 12, (0.5 + dv) * 12) >= THR
        P.append(exceedance_probability(fg, 2.0, 0.5, 12, 0.35, 30, rng))
        Pn.append(exceedance_probability(fg, 2.0, 0.5, 12, 0.05, 30, rng))
        O.append(o)
    P, Pn, O = np.array(P), np.array(Pn), np.array(O, dtype=float)
    m = (P > 0) | (Pn > 0) | (O > 0)
    print(f"Brier: calibrated ensemble {brier(P[m], O[m]):.3f}, over-confident {brier(Pn[m], O[m]):.3f}")
    rel = reliability(P[m], O[m])
    fig3, ax3 = plt.subplots(figsize=(4.8, 4.4))
    ax3.plot([0, 1], [0, 1], "k:")
    ax3.plot([r[0] for r in rel], [r[1] for r in rel], "o-", label="calibrated ensemble")
    reln = reliability(Pn[m], O[m])
    ax3.plot([r[0] for r in reln], [r[1] for r in reln], "s-", label="over-confident ensemble")
    ax3.set_xlabel("forecast probability")
    ax3.set_ylabel("observed frequency")
    ax3.legend()
    ax3.grid(alpha=0.3)
    ax3.set_title("Reliability of exceedance probabilities")
    fig3.tight_layout()
    fig3.savefig(OUT / "reliability.png", dpi=110)

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py` change the block size in `local_vectors` from 32 to 48 and compare the CSI at 60 minutes of the local forecast on `TWO_REGION`. Predict which direction it moves before you run it.
2. **Stretch.** Run `nowcast_curves(GROWTH, thr)` for thr = 5, 10 and 15 and plot the local CSI against lead time on one graph. Which threshold is hardest and why?
3. **Explain.** Explain why a storm that starts at 30 minutes cannot be found by advection, and sketch one extra source of information (not in the radar picture) that a forecaster could use.

## Assessment
- **A. Conceptual.** Why can advection never forecast a storm that starts after the forecast time? And why does a rotating pattern break a single-vector forecast?
- **B. Numerical.** A cell is moved in 12 steps of 0.5 pixels. How much blur (in pixels squared) is added in total? How much would one shift of 6 pixels add?
- **C. Graph and simulation interpretation.** In the growth and decay panel of the skill curves, at what lead time does each method fall below CSI 0.5 (10 mm/h)? What does that mean for a user?
- **D. Design and optimisation challenge.** Design: you must publish a 60-minute forecast for a field with two regions that move differently and cells that grow and fade. Choose the motion method and the longest lead time you would show as a single picture, and say what you would show beyond that.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # ---- building blocks
    f = L.scene_field([(50, 50, 5.0, 22.0, 0.0, 0.0, 0.08)], 12)
    checks.append(("a cell growing 8 % per frame has 22 x 1.08^12 = 55.4 mm/h at its centre after 12 frames",
                   abs(f[50, 50] - 22.0 * 1.08 ** 12) < 1e-9 and abs(22.0 * 1.08 ** 12 - 55.4) < 0.05))
    a = np.arange(16, dtype=float).reshape(4, 4)
    xs = np.array([[1.0, 1.5, 3.0]])
    ys = np.array([[2.0, 2.0, 3.0]])
    checks.append(("sampling at whole pixels returns the pixel, at a half pixel the average",
                   np.allclose(L.sample(a, xs, ys), [[9.0, 9.5, 15.0]])))
    checks.append(("sampling far outside the picture gives zero rain", float(L.sample(a, np.array([[50.0]]), np.array([[50.0]]))[0, 0]) == 0.0))

    # ---- global versus local motion on the two-region scene
    sc = L.TWO_REGION
    f0, f1 = L.scene_field(sc, -1), L.scene_field(sc, 0)
    gu, gv = L.global_motion(f0, f1)
    checks.append(("the single global vector is a useless compromise (-0.08, 1.10) px/frame; neither region moves like that",
                   abs(gu + 0.08) < 0.05 and abs(gv - 1.10) < 0.05))
    xc, yc, U, V = L.local_vectors(f0, f1)
    west = [(1, 0), (3, 0), (1, 1), (3, 1)]
    east = [(1, 4), (3, 4)]
    okw = all(abs(U[i, j] - 2.0) < 0.1 and abs(V[i, j] - 0.5) < 0.1 for i, j in west)
    oke = all(abs(U[i, j] + 1.5) < 0.1 and abs(V[i, j] - 1.5) < 0.1 for i, j in east)
    checks.append(("local vectors recover (2.0, 0.5) in the west and (-1.5, 1.5) in the east within 0.1 px/frame", bool(okw and oke)))
    checks.append(("blocks without rain have no vector (NaN)", bool(np.isnan(U[:, 2]).all())))
    checks.append(("global motion of two identical pictures is zero", max(abs(v) for v in L.global_motion(f1, f1)) < 1e-3))
    Up, Vp = L.vector_field(xc, yc, U, V)
    checks.append(("the pixel vector field has the size of the picture", Up.shape == (100, 100) and not np.isnan(Up).any() and not np.isnan(Vp).any()))
    z = L.vector_field(*L.local_vectors(np.zeros((100, 100)), np.zeros((100, 100))))
    checks.append(("a picture without rain gives zero vectors, not a crash", float(np.abs(z[0]).max()) == 0.0 and float(np.abs(z[1]).max()) == 0.0))
    checks.append(("advecting by 0 frames changes nothing", np.allclose(L.advect(f1, Up, Vp, 0), f1)))
    checks.append(("a uniform vector field gives the same result as a plain shift",
                   np.allclose(L.advect(f1, 2.0, 0.5, 6), L.shift_field(f1, 12.0, 3.0), atol=1e-9)))

    ob30 = L.scene_field(sc, 6)
    rm = (L.rmse(f1, ob30), L.rmse(L.advect(f1, gu, gv, 6), ob30), L.rmse(L.advect(f1, Up, Vp, 6), ob30))
    checks.append(("RMSE at 30 min: persistence 7.04, global 6.33, local 0.20 mm/h",
                   abs(rm[0] - 7.04) < 0.02 and abs(rm[1] - 6.33) < 0.02 and abs(rm[2] - 0.20) < 0.02))
    cv2 = L.nowcast_curves(sc)
    checks.append(("two regions, CSI at 30 min: global 0.13, local at least 0.99 (local vectors do better)",
                   abs(cv2["global"][5] - 0.13) < 0.01 and cv2["local"][5] >= 0.99))
    checks.append(("two regions, CSI at 60 min: local 0.90, global and persistence below 0.01",
                   abs(cv2["local"][11] - 0.90) < 0.01 and cv2["global"][11] < 0.01 and cv2["persistence"][11] < 0.01))
    cv1 = L.nowcast_curves(L.ONE_REGION)
    checks.append(("one region only: a global vector is already good (CSI at 60 min at least 0.95, about as good as local)",
                   cv1["global"][11] >= 0.95 and cv1["local"][11] >= 0.95 and abs(cv1["global"][11] - cv1["local"][11]) < 0.03))

    # ---- growth and decay; skill against lead time
    cvg = L.nowcast_curves(L.GROWTH)
    loc = cvg["local"]
    checks.append(("with growth and decay the local CSI never rises with lead time and drops from 0.91 (5 min) to 0.38 (60 min)",
                   all(loc[i + 1] <= loc[i] + 1e-12 for i in range(11)) and abs(loc[0] - 0.91) < 0.01 and abs(loc[11] - 0.38) < 0.01))
    checks.append(("persistence and global CSI also never rise with lead time",
                   all(cvg[k][i + 1] <= cvg[k][i] + 1e-12 for k in ("persistence", "global") for i in range(11))))
    checks.append(("two regions without growth: CSI at 60 min is lower than at 5 min (errors in the vectors add up)", cv2["local"][11] < cv2["local"][0]))
    ob12, ob4 = L.scene_field(L.GROWTH, 12), L.scene_field(L.GROWTH, 4)
    pure12 = L.csi(L.scene_field(L.without_growth(L.GROWTH), 12), ob12)
    pure4 = L.csi(L.scene_field(L.without_growth(L.GROWTH), 4), ob4)
    checks.append(("even with the true motion, ignoring growth costs skill: CSI 0.64 at 20 min and 0.24 at 60 min",
                   abs(pure4 - 0.636) < 0.01 and abs(pure12 - 0.236) < 0.01))
    checks.append(("with no growth the true motion gives CSI 1.0 at 60 min", L.csi(L.scene_field(L.without_growth(L.GROWTH), 12), L.scene_field(L.without_growth(L.GROWTH), 12)) == 1.0))
    checks.append(("lead time where CSI first drops below 0.5: persistence 10 min, global 15 min, local 45 min",
                   L.first_below(cvg["persistence"]) == 10 and L.first_below(cvg["global"]) == 15 and L.first_below(loc) == 45))
    checks.append(("first_below returns None when the curve never falls below the level", L.first_below([0.9, 0.8, 0.7]) is None))

    # ---- disc model for skill with displacement and growth
    s, pk, g, e = 6.0, 30.0, 0.05, 0.4
    r0 = s * math.sqrt(2 * math.log(pk / L.THR))
    good = True
    for k in (4, 8, 12):
        rt = s * math.sqrt(2 * math.log(pk * (1 + g) ** k / L.THR))
        fc = L.scene_field([(40, 50, s, pk, 0, 0, 0)], 0)
        ob = L.scene_field([(40 + e * k, 50, s, pk, 0, 0, g)], k)
        good &= abs(L.csi(fc, ob) - L.disc_overlap_csi(r0, rt, e * k)) < 0.02
    checks.append(("the two-disc formula with different radii matches the grid CSI at 20, 40 and 60 min", bool(good)))
    checks.append(("two discs: equal and on top of each other give 1, far apart 0, a small disc inside a big one gives the area ratio",
                   abs(L.disc_overlap_csi(5.0, 5.0, 0.0) - 1.0) < 1e-12 and L.disc_overlap_csi(5.0, 5.0, 10.0) == 0.0
                   and abs(L.disc_overlap_csi(2.0, 4.0, 1.0) - 0.25) < 1e-12 and L.disc_overlap_csi(0.0, 4.0, 1.0) == 0.0))

    # ---- numerical diffusion: 12 small shifts versus one big shift (worked example)
    cell = L.scene_field([(20, 50, 2.5, 30.0, 0, 0, 0)])
    rep = L.advect_repeated(cell, 0.7, 0.0, 12)
    one = L.shift_field(cell, 8.4, 0.0)
    v0 = L.second_moment_x(cell)
    checks.append(("worked example: each 0.7-pixel step adds f(1-f) = 0.21 pixel^2 of blur, so 12 steps add 2.52 and one 8.4-pixel shift only 0.24",
                   abs(L.second_moment_x(rep) - v0 - 2.52) < 0.01 and abs(L.second_moment_x(one) - v0 - 0.24) < 0.01
                   and abs(12 * 0.7 * 0.3 - 2.52) < 1e-12 and abs(0.4 * 0.6 - 0.24) < 1e-12))
    checks.append(("peak of a 30 mm/h cell after 60 min: 12 small steps 24.99, one step 29.08 (rep < one < start)",
                   abs(rep.max() - 24.99) < 0.02 and abs(one.max() - 29.08) < 0.02 and rep.max() < one.max() < cell.max()))
    checks.append(("both ways conserve total rain", abs(rep.sum() / cell.sum() - 1) < 1e-9 and abs(one.sum() / cell.sum() - 1) < 1e-9))

    # ---- ensemble and reliability
    rng = np.random.default_rng(1)
    fg = L.scene_field(L.ONE_REGION, 0)
    P, Pn, O = [], [], []
    for _ in range(40):
        du, dv = rng.normal(0, 0.35, 2)
        O.append(L.shift_field(fg, (2.0 + du) * 12, (0.5 + dv) * 12) >= L.THR)
        P.append(L.exceedance_probability(fg, 2.0, 0.5, 12, 0.35, 30, rng))
        Pn.append(L.exceedance_probability(fg, 2.0, 0.5, 12, 0.05, 30, rng))
    P, Pn, O = np.array(P), np.array(Pn), np.array(O, dtype=float)
    m = (P > 0) | (Pn > 0) | (O > 0)
    det = np.broadcast_to((L.shift_field(fg, 24.0, 6.0) >= L.THR).astype(float), O.shape)
    bs_p, bs_n, bs_d = L.brier(P[m], O[m]), L.brier(Pn[m], O[m]), L.brier(det[m], O[m])
    checks.append(("Brier score (lower is better): calibrated ensemble 0.140 < over-confident 0.179 < single answer 0.199",
                   abs(bs_p - 0.140) < 0.003 and abs(bs_n - 0.179) < 0.003 and abs(bs_d - 0.199) < 0.003 and bs_p < bs_n < bs_d))

    def wgap(rel):
        tot = sum(r[2] for r in rel)
        return sum(abs(r[0] - r[1]) * r[2] for r in rel if r[2] > 0) / tot

    checks.append(("reliability: the calibrated ensemble has a count-weighted gap below 0.03, the over-confident one above 0.1",
                   wgap(L.reliability(P[m], O[m])) < 0.03 and wgap(L.reliability(Pn[m], O[m])) > 0.1))
    p0 = L.exceedance_probability(fg, 2.0, 0.5, 12, 0.0, 5, np.random.default_rng(0))
    checks.append(("with zero wind uncertainty every member agrees: probabilities are only 0 or 1",
                   bool(np.all((p0 == 0.0) | (p0 == 1.0))) and np.array_equal(p0 >= 0.5, L.shift_field(fg, 24.0, 6.0) >= L.THR)))
    checks.append(("probabilities stay between 0 and 1", float(P.min()) >= 0.0 and float(P.max()) <= 1.0))

    # ---- failure cases
    ob0 = L.scene_field(sc, 12)
    ob1 = L.with_new_storm(ob0, 6, 12)
    fc = L.advect(f1, Up, Vp, 12)
    yy, xx = np.mgrid[0:100, 0:100]
    reg = (xx - 55) ** 2 + (yy - 10) ** 2 <= 12 ** 2
    hits = int(((fc >= L.THR) & (ob1 >= L.THR) & reg).sum())
    nobs = int(((ob1 >= L.THR) & reg).sum())
    checks.append(("a storm that starts at 30 min: 0 of its 193 rainy pixels are forecast; CSI falls from 0.90 to 0.72",
                   hits == 0 and nobs == 193 and abs(L.csi(fc, ob0) - 0.896) < 0.003 and abs(L.csi(fc, ob1) - 0.724) < 0.003))
    r0f, r1f = L.rotating_scene_fields(-1), L.rotating_scene_fields(0)
    ru, rv = L.global_motion(r0f, r1f)
    rob = L.rotating_scene_fields(12)
    ru_l, rv_l = L.vector_field(*L.local_vectors(r0f, r1f))
    c_pers, c_glob, c_loc = L.csi(r1f, rob), L.csi(L.advect(r1f, ru, rv, 12), rob), L.csi(L.advect(r1f, ru_l, rv_l, 12), rob)
    checks.append(("a rotating pattern has no mean motion: global vector about 0, so global = persistence = CSI 0.07; local 0.08 (all below 0.1)",
                   abs(ru) < 1e-3 and abs(rv) < 1e-3 and abs(c_pers - 0.073) < 0.003 and abs(c_glob - c_pers) < 1e-3
                   and abs(c_loc - 0.080) < 0.003 and max(c_pers, c_glob, c_loc) < 0.1))
    checks.append(("CSI is undefined (nan) for two pictures without rain", math.isnan(L.csi(np.zeros((5, 5)), np.zeros((5, 5))))))
    mp = {t: L.nowcast_curves(L.GROWTH, t) for t in (5.0, 15.0)}
    checks.append(("mini-project: lead time below CSI 0.5 at 5 mm/h is persistence 15, global 15, local never (CSI 0.62 at 60 min)",
                   L.first_below(mp[5.0]["persistence"]) == 15 and L.first_below(mp[5.0]["global"]) == 15
                   and L.first_below(mp[5.0]["local"]) is None and abs(mp[5.0]["local"][11] - 0.62) < 0.01))
    checks.append(("mini-project: at 10 mm/h it is 10, 15 and 45 minutes; at 15 mm/h it is 10, 10 and 25 minutes",
                   L.first_below(cvg["persistence"]) == 10 and L.first_below(cvg["global"]) == 15 and L.first_below(cvg["local"]) == 45
                   and L.first_below(mp[15.0]["persistence"]) == 10 and L.first_below(mp[15.0]["global"]) == 10 and L.first_below(mp[15.0]["local"]) == 25))
    r12 = L.advect_repeated(cell, 0.5, 0.0, 12)
    checks.append(("exercise: 12 steps of 0.5 pixel add 12 x 0.25 = 3.0 pixel^2 of blur; one 6-pixel shift adds 0",
                   abs(L.second_moment_x(r12) - v0 - 3.0) < 0.01 and abs(L.second_moment_x(L.shift_field(cell, 6.0, 0.0)) - v0) < 1e-9))
    r0g = 6.0 * math.sqrt(2 * math.log(30.0 / 10.0))
    rg = 6.0 * math.sqrt(2 * (math.log(3.0) + 12 * math.log(1.05)))
    checks.append(("app: growth of 5 % per frame, error 6 km/h at 60 min gives CSI 0.43; error 3 km/h gives 0.61; growth alone 0.65",
                   abs(L.disc_overlap_csi(r0g, rg, 6.0) - 0.430) < 0.002 and abs(L.disc_overlap_csi(r0g, rg, 3.0) - 0.614) < 0.002
                   and abs(L.disc_overlap_csi(r0g, rg, 0.0) - 0.652) < 0.002 and abs(r0g - 8.89) < 0.01 and abs(rg - 11.01) < 0.01))
    top = L.reliability(P[m], O[m])[-1]
    checks.append(("roadmap: the calibrated ensemble still has a gap of 0.12 in its highest bin (more than 1000 pixels)",
                   top[2] > 1000 and abs(abs(top[0] - top[1]) - 0.117) < 0.005))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
